In [ ]:
import pandas as pd
import numpy as np
import optuna

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelEncoder

hello world


In [ ]:
dataset_path = "../data/train.csv"

data = pd.read_csv(dataset_path, index_col=0)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(data["Status"])
X = data.drop("Status", axis=1)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
categorical_cols = [
    c for c in X_train.columns
    if X_train[c].dtype == "object"
]

numerical_cols = [
    c for c in X_train.columns
    if X_train[c].dtype in ["int64", "float64"]
]

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numerical_transformer = SimpleImputer(strategy='constant')

# Preprocessing for categorical data
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=1000, 
    learning_rate=0.01, 
    random_state=0
)

from sklearn.metrics import log_loss

# Bundle preprocessing and modeling code in a pipeline
my_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', model)
                             ])

# X_transformed = my_pipeline.named_steps['preprocessor'].transform(X_valid)

# # If the output is a sparse matrix, convert to array to view it
# # Get names of all features after preprocessing
# feature_names = my_pipeline.named_steps['preprocessor'].get_feature_names_out()

# # Create a DataFrame to make it readable
# transformed_df = pd.DataFrame(X_transformed, columns=feature_names)
# print(transformed_df.head())
# Preprocessing of training data, fit model 
my_pipeline.fit(X_train, y_train)

# Preprocessing of validation data, get predictions
preds = my_pipeline.predict_proba(X_valid)

# Evaluate the model
score = log_loss(y_valid, preds)
print('log_loss:', score)

test_preds_proba = my_pipeline.predict_proba(X_test)

# 2. Create columns for each class (e.g., Status_0, Status_1, Status_2)
class_names = [f"Status_{i}" for i in range(test_preds_proba.shape[1])]
output = pd.DataFrame(test_preds_proba, columns=class_names)

# 3. Add the ID column and save
output.insert(0, 'id', X_test_ids)
output.to_csv('submission_probs.csv', index=False)